# Gold: lab_test_trends

Builds `azuredb_caresync.gold.lab_test_trends` (one row per test type per month) from the silver layer per the STTM `sttm_silver_to_gold.xlsx`.

* **Driving table:** `silver.lab_results` grouped by `test_name`, year/month of `test_date`
* **Joins:** LEFT JOIN `silver.calendar` on the first day of the month's `date_key`
* **Window:** `LAG(abnormal_rate_pct)` per test ordered by year, month for month-over-month trend
* **Load type:** FULL rebuild — `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS azuredb_caresync.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE azuredb_caresync.gold.lab_test_trends
COMMENT 'Gold: one row per lab test type per month with abnormal/critical rates and month-over-month trend'
AS
WITH monthly AS (
  -- Lab results aggregated to test_name x month grain
  SELECT
    test_name,
    year(test_date)                        AS year,
    month(test_date)                       AS month,
    COUNT(*)                               AS total_tests,
    COUNT_IF(result_status = 'Abnormal')   AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')   AS critical_test_count
  FROM azuredb_caresync.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    *,
    CAST(abnormal_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
lagged AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  l.test_name,
  CAST(l.year AS INT)                                                       AS year,
  CAST(l.month AS INT)                                                      AS month,
  c.month_name,
  c.quarter_name,
  CAST(l.total_tests AS INT)                                                AS total_tests,
  CAST(l.abnormal_test_count AS INT)                                        AS abnormal_test_count,
  CAST(l.critical_test_count AS INT)                                        AS critical_test_count,
  l.abnormal_rate_pct,
  l.critical_rate_pct,
  CAST(l.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                      AS prev_month_abnormal_rate_pct,
  CAST(l.abnormal_rate_pct - l.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct  -- percentage points, NULL for first month
FROM lagged l
LEFT JOIN azuredb_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(make_date(l.year, l.month, 1), 'yyyyMMdd') AS INT);

In [0]:
updated_df = spark.table("azuredb_caresync.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))